# Specimen 04 — Concurrent Research Assistant

Goal: turn Phase 5 Specimen 04 into its concurrent form -- the actual final project. Same planner, same critic, same writer. The sequential `for` loop over researchers becomes `asyncio.gather`; `ResearchStore` gets Specimen 02's lock; the supervisor gains Specimen 03's retry-once-on-failure and per-task timeout. This is a conversion of that exact codebase, not a new build.

In [1]:
import os
import datetime
import json
import time
import asyncio
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.AsyncAnthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
MODEL = 'claude-opus-5'

INPUT_PRICE_PER_MTOK = 5.00
OUTPUT_PRICE_PER_MTOK = 25.00

def call_cost(usage):
    return (usage.input_tokens / 1_000_000 * INPUT_PRICE_PER_MTOK) + (usage.output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MTOK)

# Caps how many requests are in flight at once, regardless of how many coroutines are scheduled --
# asyncio.gather alone fires every call simultaneously, which is the fastest way to get rate-limited.
_concurrency_limit = asyncio.Semaphore(5)

async def call_model(messages, tools=None, max_tokens=1200, output_schema=None):
    kwargs = dict(model=MODEL, max_tokens=max_tokens, messages=messages, thinking={"type": "disabled"})
    if tools:
        kwargs['tools'] = tools
    if output_schema:
        kwargs['output_config'] = {"format": {"type": "json_schema", "schema": output_schema}}
    async with _concurrency_limit:
        return await client.messages.create(**kwargs)


`call_model` is now `async` and built on `AsyncAnthropic` -- `asyncio.gather` needs real awaitable coroutines to get genuine concurrency; wrapping the sync client wouldn't actually overlap requests. A `Semaphore` caps in-flight requests at 5, since `gather` alone will fire every call at once. Everything else carries forward from Phase 3-5: `thinking` disabled by default, schema-enforced JSON handoffs via `output_schema`, and a default `max_tokens` of 1200 -- Phase 5 found 800 too tight for planner/researcher-style structured output, where the model kept writing past the budget and breaking the JSON mid-generation. One more Phase 5 finding worth remembering here: `output_config`'s JSON schema does not support `maxItems` on arrays -- bound response length through the prompt (ask for an exact count, not "at most"), not the schema.

## 1. Carry the planner over unchanged

Breaking a broad question into subtopics is still one sequential call -- there's nothing to parallelize about it. Reuse Phase 5's planner as-is.

In [2]:
PLANNER_SYSTEM = (
    "You are a planning agent. Given a broad research question, break it into independent "
    "subtopics that can each be researched separately, without needing another subtopic's findings."
)

PLAN_SCHEMA = {
    "type": "object",
    "properties": {
        "subtopics": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {"key": {"type": "string"}, "description": {"type": "string"}},
                "required": ["key", "description"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["subtopics"],
    "additionalProperties": False,
}

async def run_planner(question, max_subtopics=5):
    response = await call_model(
        messages=[{"role": "user", "content": (
            f"{PLANNER_SYSTEM}\n\nBroad question: {question}\n\n"
            f"Break this into at most {max_subtopics} independent subtopics. Each 'key' should be a short slug."
        )}],
        output_schema=PLAN_SCHEMA,
        max_tokens=2000,
    )
    text = ''.join(b.text for b in response.content if b.type == 'text')
    subtopics = json.loads(text)["subtopics"]
    if len(subtopics) > max_subtopics:
        subtopics = subtopics[:max_subtopics]
    return subtopics, response.usage

# Same question as Phase 5 Specimen 04, so the wall-clock comparison in step 7 is a fair one.
QUESTION = "Compare four popular Python web frameworks -- Django, Flask, FastAPI, and Tornado -- for building a REST API that needs to serve 10,000 requests per second."
subtopics, planner_usage = await run_planner(QUESTION)
for s in subtopics:
    print(f"{s['key']}: {s['description']}")

django-rest-api-profile: Research Django (with Django REST Framework) as a REST API platform: its architecture (WSGI/ASGI support, sync vs async views), request-handling overhead from middleware and ORM, published benchmark throughput and latency figures for JSON endpoints, deployment stack options (gunicorn/uvicorn workers, Daphne), caching and connection-pooling strategies, and documented cases or techniques for pushing it toward very high request rates (order of 10k rps). Note memory/CPU cost per worker and known bottlenecks.
flask-rest-api-profile: Research Flask (plus extensions like Flask-RESTful/Flask-Smorest) as a REST API platform: WSGI synchronous model, threading vs gevent/eventlet vs multi-process serving, per-request overhead, published benchmark throughput and latency numbers for JSON endpoints, serving stack choices (gunicorn workers, uWSGI, meinheld), and practical ceilings and tuning techniques for high-throughput scenarios near 10k rps.
fastapi-rest-api-profile: Resea

## 2. Carry `ResearchStore` over with Specimen 02's lock

Same class shape as Phase 5, now with the `asyncio.Lock` from Specimen 02 wrapped around `get`/`set`.

In [3]:
class LockedResearchStore:
    def __init__(self):
        self._data = {}
        self._lock = asyncio.Lock()

    async def set(self, key, value):
        async with self._lock:
            self._data[key] = value

    async def get(self, key):
        async with self._lock:
            return self._data.get(key)

    def items(self):
        return list(self._data.items())

    def __len__(self):
        return len(self._data)

store = LockedResearchStore()
print("Locked store ready -- same shape as Phase 5's ResearchStore, get/set now serialize on an asyncio.Lock.")

Locked store ready -- same shape as Phase 5's ResearchStore, get/set now serialize on an asyncio.Lock.


## 3. Convert the sequential critic-gated research loop to `asyncio.gather`

Phase 5 ran one subtopic's researcher-then-critic cycle at a time in a `for` loop. Here, launch all subtopics' critic-gated research coroutines together and `await asyncio.gather(...)` them -- each one still runs its own internal researcher→critic→revise sequence, but the subtopics themselves now run concurrently with each other.

In [4]:
log_lines = []

def log_event(trace_id, message):
    ts = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    log_lines.append({"ts": ts, "trace_id": trace_id, "message": message})
    print(f"[{ts}] [{trace_id}] {message}")

RESEARCHER_SYSTEM = (
    "You are a research agent. Research the given subtopic using your own knowledge and produce a "
    "structured research note. The subtopic may be broad, but the note must stay short regardless: "
    "write a summary of exactly 1-2 sentences, exactly 4 key_facts as short one-line items (pick the "
    "4 most important -- do not try to cover every aspect mentioned in the subtopic), and exactly 2 "
    "open_questions. Brevity matters more than exhaustive coverage."
)

RESEARCH_NOTE_SCHEMA = {
    "type": "object",
    "properties": {
        "summary": {"type": "string"},
        "key_facts": {"type": "array", "items": {"type": "string"}},
        "open_questions": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["summary", "key_facts", "open_questions"],
    "additionalProperties": False,
}

async def run_researcher(subtopic_description, feedback=None):
    prompt = f"Subtopic: {subtopic_description}"
    if feedback:
        prompt += f"\n\nA critic reviewed your previous note and said: {feedback}\nProduce a revised note that addresses this."
    response = await call_model(
        messages=[{"role": "user", "content": f"{RESEARCHER_SYSTEM}\n\n{prompt}"}],
        output_schema=RESEARCH_NOTE_SCHEMA,
        max_tokens=1200,
    )
    text = ''.join(b.text for b in response.content if b.type == 'text')
    return json.loads(text), response.usage

CRITIC_SYSTEM = (
    "You are a critic agent. Review a research note against this rubric. Judge ONLY against the "
    "rubric -- return 'approve' only if every criterion is met, otherwise 'revise' with a specific reason."
)

VERDICT_SCHEMA = {
    "type": "object",
    "properties": {
        "verdict": {"type": "string", "enum": ["approve", "revise"]},
        "reason": {"type": "string"},
    },
    "required": ["verdict", "reason"],
    "additionalProperties": False,
}

RESEARCH_RUBRIC = (
    "1. 'summary' must be non-empty and at least one full sentence.\n"
    "2. 'key_facts' must contain at least 3 distinct, specific facts (not vague restatements of the summary).\n"
    "3. 'open_questions' must contain at least 1 genuine open question."
)

async def run_research_critic(note):
    response = await call_model(
        messages=[{"role": "user", "content": (
            f"{CRITIC_SYSTEM}\n\nRubric:\n{RESEARCH_RUBRIC}\n\nResearch note (JSON):\n{json.dumps(note)}"
        )}],
        output_schema=VERDICT_SCHEMA,
    )
    text = ''.join(b.text for b in response.content if b.type == 'text')
    return json.loads(text), response.usage

async def research_with_critic(subtopic, max_revisions=2):
    trace_id = subtopic["key"]
    log_event(trace_id, "subtopic research started")
    history = []
    feedback = None
    note, usages = None, []
    for attempt_num in range(1, max_revisions + 2):
        log_event(trace_id, f"researcher call started (attempt {attempt_num})")
        note, r_usage = await run_researcher(subtopic["description"], feedback)
        usages.append(r_usage)
        log_event(trace_id, f"researcher call finished (attempt {attempt_num})")
        log_event(trace_id, "critic call started")
        verdict, c_usage = await run_research_critic(note)
        usages.append(c_usage)
        log_event(trace_id, f"critic call finished -- verdict: {verdict['verdict']}")
        history.append({"attempt": attempt_num, "verdict": verdict})
        if verdict["verdict"] == "approve":
            log_event(trace_id, "subtopic research approved")
            return note, history, usages, False
        feedback = verdict["reason"]
    log_event(trace_id, "subtopic research capped without approval")
    return note, history, usages, True

async def research_subtopic_and_store(subtopic, store):
    note, history, usages, capped = await research_with_critic(subtopic)
    await store.set(subtopic["key"], note)
    return subtopic["key"], history, usages, capped

# Launch every subtopic's researcher-critic cycle at once -- each one is still sequential
# *within itself*, but the subtopics now run alongside each other instead of one at a time.
gather_results = await asyncio.gather(*[research_subtopic_and_store(s, store) for s in subtopics])

print()
for key, history, usages, capped in gather_results:
    status = "capped with caveat" if capped else "approved"
    print(f"[{key}] {status} after {len(history)} attempt(s)")
print(f"\nStore now holds {len(store)} entries")

[13:24:01.124] [django-rest-api-profile] subtopic research started
[13:24:01.124] [django-rest-api-profile] researcher call started (attempt 1)
[13:24:01.126] [flask-rest-api-profile] subtopic research started
[13:24:01.126] [flask-rest-api-profile] researcher call started (attempt 1)
[13:24:01.127] [fastapi-rest-api-profile] subtopic research started
[13:24:01.127] [fastapi-rest-api-profile] researcher call started (attempt 1)
[13:24:01.128] [tornado-rest-api-profile] subtopic research started
[13:24:01.128] [tornado-rest-api-profile] researcher call started (attempt 1)
[13:24:01.128] [cross-framework-benchmarks-and-scaling] subtopic research started
[13:24:01.128] [cross-framework-benchmarks-and-scaling] researcher call started (attempt 1)


[13:24:13.770] [tornado-rest-api-profile] researcher call finished (attempt 1)
[13:24:13.770] [tornado-rest-api-profile] critic call started


[13:24:15.568] [fastapi-rest-api-profile] researcher call finished (attempt 1)
[13:24:15.568] [fastapi-rest-api-profile] critic call started
[13:24:15.638] [flask-rest-api-profile] researcher call finished (attempt 1)
[13:24:15.638] [flask-rest-api-profile] critic call started


[13:24:15.802] [django-rest-api-profile] researcher call finished (attempt 1)
[13:24:15.803] [django-rest-api-profile] critic call started


[13:24:18.136] [tornado-rest-api-profile] critic call finished -- verdict: approve
[13:24:18.136] [tornado-rest-api-profile] subtopic research approved


[13:24:20.470] [django-rest-api-profile] critic call finished -- verdict: approve
[13:24:20.471] [django-rest-api-profile] subtopic research approved
[13:24:20.472] [fastapi-rest-api-profile] critic call finished -- verdict: approve
[13:24:20.472] [fastapi-rest-api-profile] subtopic research approved
[13:24:20.473] [flask-rest-api-profile] critic call finished -- verdict: approve
[13:24:20.473] [flask-rest-api-profile] subtopic research approved
[13:24:20.655] [cross-framework-benchmarks-and-scaling] researcher call finished (attempt 1)
[13:24:20.655] [cross-framework-benchmarks-and-scaling] critic call started


[13:24:25.669] [cross-framework-benchmarks-and-scaling] critic call finished -- verdict: approve
[13:24:25.670] [cross-framework-benchmarks-and-scaling] subtopic research approved

[django-rest-api-profile] approved after 1 attempt(s)
[flask-rest-api-profile] approved after 1 attempt(s)
[fastapi-rest-api-profile] approved after 1 attempt(s)
[tornado-rest-api-profile] approved after 1 attempt(s)
[cross-framework-benchmarks-and-scaling] approved after 1 attempt(s)

Store now holds 5 entries


## 4. Add structured logging with a trace ID per subtopic

Per the final-project design decision: every agent transition (researcher attempt N for subtopic X, critic verdict for subtopic X, writer started) gets a timestamped log line tagged with that subtopic's trace ID. With concurrent agents interleaving, you can no longer read the log top-to-bottom and assume order -- the trace ID is what lets you filter back down to one subtopic's story after the fact.

In [5]:
print("All log lines, in real execution order (interleaved across subtopics):")
for line in log_lines[:12]:
    print(f"  [{line['ts']}] [{line['trace_id']}] {line['message']}")
print("  ...")

sample_trace = subtopics[0]["key"]
print(f"\nFiltered back to just '{sample_trace}' using its trace ID:")
for line in log_lines:
    if line["trace_id"] == sample_trace:
        print(f"  [{line['ts']}] [{line['trace_id']}] {line['message']}")

All log lines, in real execution order (interleaved across subtopics):
  [13:24:01.124] [django-rest-api-profile] subtopic research started
  [13:24:01.124] [django-rest-api-profile] researcher call started (attempt 1)
  [13:24:01.126] [flask-rest-api-profile] subtopic research started
  [13:24:01.126] [flask-rest-api-profile] researcher call started (attempt 1)
  [13:24:01.127] [fastapi-rest-api-profile] subtopic research started
  [13:24:01.127] [fastapi-rest-api-profile] researcher call started (attempt 1)
  [13:24:01.128] [tornado-rest-api-profile] subtopic research started
  [13:24:01.128] [tornado-rest-api-profile] researcher call started (attempt 1)
  [13:24:01.128] [cross-framework-benchmarks-and-scaling] subtopic research started
  [13:24:01.128] [cross-framework-benchmarks-and-scaling] researcher call started (attempt 1)
  [13:24:13.770] [tornado-rest-api-profile] researcher call finished (attempt 1)
  [13:24:13.770] [tornado-rest-api-profile] critic call started
  ...

Filte

## 5. Add per-subtopic timeout and retry-once-on-failure

Reuse Specimen 03's pattern. Confirm a simulated single-subtopic failure gets retried and doesn't stall or take down the other subtopics running alongside it.

In [6]:
_fail_once_tracker = {}

async def maybe_inject_failure(trace_id):
    if trace_id == "simulated-flaky-subtopic" and _fail_once_tracker.get(trace_id, 0) == 0:
        _fail_once_tracker[trace_id] = 1
        raise RuntimeError("Simulated transient failure (e.g. a dropped connection)")

async def research_subtopic_and_store_flaky(subtopic, store):
    await maybe_inject_failure(subtopic["key"])
    return await research_subtopic_and_store(subtopic, store)

async def research_subtopic_supervised(subtopic, store, timeout=30, max_retries=1):
    trace_id = subtopic["key"]
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            await asyncio.wait_for(research_subtopic_and_store_flaky(subtopic, store), timeout=timeout)
            return {"key": trace_id, "status": "ok", "attempts": attempt + 1}
        except Exception as e:
            last_error = e
            log_event(trace_id, f"supervisor caught failure on attempt {attempt + 1}: {e}")
    return {"key": trace_id, "status": "failed", "error": str(last_error), "attempts": max_retries + 1}

flaky_subtopic = {"key": "simulated-flaky-subtopic", "description": "A short, simple test subtopic used only to demonstrate the supervisor's retry-on-failure behavior."}

supervised_results = await asyncio.gather(research_subtopic_supervised(flaky_subtopic, store, timeout=30))
for r in supervised_results:
    print(f"[{r['key']}] {r['status']} after {r['attempts']} attempt(s)")

[13:24:25.708] [simulated-flaky-subtopic] supervisor caught failure on attempt 1: Simulated transient failure (e.g. a dropped connection)
[13:24:25.708] [simulated-flaky-subtopic] subtopic research started
[13:24:25.708] [simulated-flaky-subtopic] researcher call started (attempt 1)


[13:24:31.456] [simulated-flaky-subtopic] researcher call finished (attempt 1)
[13:24:31.457] [simulated-flaky-subtopic] critic call started


[13:24:35.568] [simulated-flaky-subtopic] critic call finished -- verdict: approve
[13:24:35.569] [simulated-flaky-subtopic] subtopic research approved
[simulated-flaky-subtopic] ok after 2 attempt(s)


## 6. Writer synthesizes the store, same as Phase 5

Unchanged from Phase 5 -- the writer only ever saw the finished store, never cared whether it was filled sequentially or concurrently.

In [7]:
WRITER_SYSTEM = (
    "You are a writer agent. You are given a set of research notes, one per subtopic, all keyed by "
    "subtopic. Synthesize them into one coherent final report that directly answers the original "
    "broad question. Use only what's in these notes -- you have not seen any of the underlying research."
)

async def run_writer(question, store, subtopics):
    notes = {}
    for s in subtopics:
        note = await store.get(s["key"])
        if note:
            notes[s["key"]] = note
    notes_blob = json.dumps(notes, indent=2)
    response = await call_model(
        messages=[{"role": "user", "content": (
            f"{WRITER_SYSTEM}\n\nOriginal question: {question}\n\nResearch notes (JSON):\n{notes_blob}"
        )}],
        max_tokens=700,
    )
    text = ''.join(b.text for b in response.content if b.type == 'text')
    return text, response.usage

# Scoped to the real `subtopics` list -- deliberately ignores the flaky demo entry from step 5.
report, writer_usage = await run_writer(QUESTION, store, subtopics)
print(report)

# Serving 10,000 Requests/Second: Django vs. Flask vs. FastAPI vs. Tornado

## Executive Summary

No mainstream Python web framework will serve 10,000 requests per second from a single process. All four candidates — Django, Flask, FastAPI, and Tornado — land in the low thousands of requests/second per worker for simple JSON endpoints, and every one of them reaches 10k rps the same way: multiple worker processes across one or more hosts, behind a reverse proxy, with caching and connection pooling in front of the database.

That said, the frameworks are not equivalent. Ranked by per-core efficiency on JSON workloads, the ordering that emerges consistently from TechEmpower and independent benchmarks is:

**FastAPI (Uvicorn/uvloop) ≳ Tornado > Flask > Django/DRF**

The practical conclusion is twofold. First, **framework choice sets your per-node efficiency and therefore your cluster size and cost, not whether 10k rps is possible.** Second, **hitting 10k rps is primarily an infrastructure a

## 7. Run it end-to-end and compare wall-clock time against Phase 5

Same broad question Phase 5 Specimen 04 used, so the comparison is fair. Report total wall-clock time, the supervisor summary (retries, failures, cost), and the speedup (or lack of one) versus the sequential run -- if concurrency didn't actually help here, that's a real finding worth reporting honestly, not a result to paper over.

In [8]:
async def run_research_assistant_concurrent(question, max_subtopics=5, max_revisions=2, max_retries=1, timeout=30):
    start = time.perf_counter()
    all_usages = []

    subtopics, planner_usage = await run_planner(question, max_subtopics=max_subtopics)
    all_usages.append(planner_usage)

    local_store = LockedResearchStore()

    async def _research_one(s):
        trace_id = s["key"]
        last_error = None
        for attempt in range(max_retries + 1):
            try:
                note, history, usages, capped = await asyncio.wait_for(
                    research_with_critic(s, max_revisions=max_revisions), timeout=timeout
                )
                await local_store.set(trace_id, note)
                return {"key": trace_id, "status": "capped" if capped else "ok", "usages": usages, "attempts": attempt + 1}
            except Exception as e:
                last_error = e
                log_event(trace_id, f"supervisor caught failure on attempt {attempt + 1}: {e}")
        return {"key": trace_id, "status": "failed", "usages": [], "attempts": max_retries + 1, "error": str(last_error)}

    subtopic_results = await asyncio.gather(*[_research_one(s) for s in subtopics])
    for r in subtopic_results:
        all_usages.extend(r["usages"])

    report, writer_usage = await run_writer(question, local_store, subtopics)
    all_usages.append(writer_usage)

    elapsed = time.perf_counter() - start
    total_cost = sum(call_cost(u) for u in all_usages)
    failed = [r["key"] for r in subtopic_results if r["status"] == "failed"]
    retried = [r["key"] for r in subtopic_results if r["attempts"] > 1]

    summary = {
        "subtopics_researched": len(subtopics),
        "total_api_calls": len(all_usages),
        "retried_subtopics": retried,
        "failed_subtopics": failed,
        "total_cost": total_cost,
        "wall_clock_seconds": elapsed,
    }
    return report, summary

print("run_research_assistant_concurrent() defined -- planner -> concurrent critic-gated researchers (locked store, logged, supervised with retry+timeout) -> writer, cost- and time-tracked end to end.")


# Fair, isolated comparison: same subtopics, same critic-gated research function,
# sequential loop vs. asyncio.gather -- nothing else in the mix.
seq_start = time.perf_counter()
for s in subtopics:
    await research_with_critic(s, max_revisions=2)
seq_elapsed = time.perf_counter() - seq_start

conc_start = time.perf_counter()
await asyncio.gather(*[research_with_critic(s, max_revisions=2) for s in subtopics])
conc_elapsed = time.perf_counter() - conc_start

print(f"\nSequential research phase ({len(subtopics)} subtopics): {seq_elapsed:.1f}s")
print(f"Concurrent research phase ({len(subtopics)} subtopics): {conc_elapsed:.1f}s")
print(f"Speedup: {seq_elapsed / conc_elapsed:.1f}x")

# The actual end-to-end deliverable, using the supervised/logged/locked pipeline built above.
final_report, run_summary = await run_research_assistant_concurrent(QUESTION)

print("\nFINAL REPORT")
print("=" * 60)
print(final_report)
print()
print("SUPERVISOR SUMMARY")
print("=" * 60)
for k, v in run_summary.items():
    print(f"{k}: {v}")

run_research_assistant_concurrent() defined -- planner -> concurrent critic-gated researchers (locked store, logged, supervised with retry+timeout) -> writer, cost- and time-tracked end to end.
[13:24:46.593] [django-rest-api-profile] subtopic research started
[13:24:46.593] [django-rest-api-profile] researcher call started (attempt 1)


[13:25:00.122] [django-rest-api-profile] researcher call finished (attempt 1)
[13:25:00.123] [django-rest-api-profile] critic call started


[13:25:05.350] [django-rest-api-profile] critic call finished -- verdict: approve
[13:25:05.350] [django-rest-api-profile] subtopic research approved
[13:25:05.350] [flask-rest-api-profile] subtopic research started
[13:25:05.350] [flask-rest-api-profile] researcher call started (attempt 1)


[13:25:18.288] [flask-rest-api-profile] researcher call finished (attempt 1)
[13:25:18.289] [flask-rest-api-profile] critic call started


[13:25:22.573] [flask-rest-api-profile] critic call finished -- verdict: approve
[13:25:22.573] [flask-rest-api-profile] subtopic research approved
[13:25:22.573] [fastapi-rest-api-profile] subtopic research started
[13:25:22.573] [fastapi-rest-api-profile] researcher call started (attempt 1)


...

[13:27:11.504] [fastapi-rest-api-performance] critic call finished -- verdict: approve
[13:27:11.504] [fastapi-rest-api-performance] subtopic research approved


[13:27:13.911] [django-rest-api-performance] critic call finished -- verdict: approve
[13:27:13.922] [django-rest-api-performance] subtopic research approved


[13:27:14.900] [high-throughput-methodology-and-infrastructure] critic call finished -- verdict: approve
[13:27:14.900] [high-throughput-methodology-and-infrastructure] subtopic research approved
[13:27:14.900] [flask-rest-api-performance] critic call finished -- verdict: approve
[13:27:14.900] [flask-rest-api-performance] subtopic research approved


[13:27:15.322] [tornado-rest-api-performance] critic call finished -- verdict: approve
[13:27:15.322] [tornado-rest-api-performance] subtopic research approved



FINAL REPORT
# Choosing a Python Web Framework for a 10,000 RPS REST API

## Executive Summary

The short answer: **at 10,000 requests per second, framework choice matters far less than architecture.** All four frameworks — Django, Flask, FastAPI, and Tornado — can reach 10k RPS, and none of them can do it on a single process. Real-world Python APIs with a database and business logic typically land in the hundreds to low thousands of requests per second *per CPU core*, so 10k RPS is fundamentally a capacity-planning and horizontal-scaling problem rather than a framework-selection problem.

That said, the frameworks differ meaningfully in how much hardware they need to get there, how well they handle concurrent I/O, and what you give up in ecosystem maturity or developer productivity. FastAPI has the strongest raw-throughput profile; Django has the strongest feature set; Flask is the simplest to reason about; Tornado is a specialist tool whose real advantage lies elsewhere.

---

## Th